In [1]:
# !pip install meteostat geopy

In [2]:
# Импорт библиотек

from meteostat import Point, Daily
from geopy.geocoders import Nominatim
from datetime import datetime
import pandas as pd
import time

In [3]:
# Загружаем данные по городам, создаем уникальный список городов

data = pd.read_csv('data/cities.csv', sep=';')
cities = set(data['City'])

In [4]:
# Обозначаем период

start = datetime(2023, 5, 22)
end = datetime(2025, 6, 2)

In [5]:
# Геокодер

geolocator = Nominatim(user_agent="weather_app")

In [6]:
# Соберем все данные

all_weather = []

for city in cities:
    try:
        location = geolocator.geocode(city)
        if location:
            print(f"Получены координаты: {city} -> {location.latitude}, {location.longitude}")
            point = Point(location.latitude, location.longitude)

            # Получаем данные
            
            data = Daily(point, start, end).fetch()
            data.reset_index(inplace=True)
            data['city'] = city
            all_weather.append(data)
        else:
            print(f"Не удалось найти координаты для {city}")
    except Exception as e:
        print(f"Ошибка с {city}: {e}")
    
    time.sleep(1)

# Объединяем всё

df_weather = pd.concat(all_weather, ignore_index=True)

# Оставим только нужные столбцы

df_weather = df_weather[['time', 'city', 'tavg']]

Получены координаты: BALOTESTI -> 46.4016758, 27.3135004
Получены координаты: SIBIU -> 45.7973912, 24.1519202
Получены координаты: IASI -> 47.1615598, 27.5837814
Получены координаты: PITESTI -> 44.8572343, 24.8719422
Получены координаты: DROBETA-TURNU SEVERI -> 44.6257835, 22.6531975
Получены координаты: CONSTANTA -> 44.1767161, 28.6507598
Получены координаты: GALATI -> 45.4338215, 28.0549395
Получены координаты: RAMNICU VALCEA -> 45.1031731, 24.3647209
Получены координаты: BRAILA -> 45.2716092, 27.9742932
Получены координаты: TIMISOARA -> 45.7538355, 21.2257474
Получены координаты: TARGOVISTE -> 44.9267709, 25.462816
Получены координаты: TARGU MURES -> 46.5446253, 24.561196
Получены координаты: BAIA MARE -> 47.6565584, 23.5719843
Получены координаты: PLOIESTI -> 44.9417468, 26.0236504
Получены координаты: SUCEAVA -> 47.5326534, 25.8345939
Получены координаты: BUCHAREST -> 44.4378654, 26.0863631
Получены координаты: CRAIOVA -> 44.3190159, 23.7965614
Получены координаты: ALBA IULIA -> 4

In [7]:
# Посчитаем среднюю температуру за последние 7 дней для каждого из городов

df_weather['time'] = pd.to_datetime(df_weather['time'])
df_weather.sort_values(['city', 'time'], inplace=True)

# Применим скользящее среднее по группам городов

df_weather['tavg_7d'] = df_weather.groupby('city')['tavg'].transform(lambda x: x.rolling(7, min_periods=1).mean().round(2))

In [8]:
# сбросим индексы и запишем данные в файл

df_weather.reset_index(inplace=True)
df_weather.to_csv('data/weather_data.csv', index=False)